# stride-zero-broadcast — ex2: detect zero-stride view, materialize via .contiguous(), report stride + storage delta

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `stride-zero-broadcast`. Running the final beacon cell reports progress against the `PyTorch: Zero-stride broadcasting` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Zero-stride broadcasting` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`stride-zero-broadcast`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "stride-zero-broadcast"
DD_SUBTOPIC = "PyTorch: Zero-stride broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Zero-stride detection → `.contiguous()` materialization

Ex1 distinguished `.expand()` (zero-stride view) from `.repeat()` (true copy). The deepening move: given an ARBITRARY tensor, detect whether any axis has stride 0, then call `.contiguous()` and verify the materialization happened.

```python
x = t.arange(3).expand(4, 3)        # shape (4, 3), stride (0, 1)
has_broadcast = any(s == 0 for s in x.stride())   # True
y = x.contiguous()                  # forces a copy
# y has stride (3, 1) and y.storage().nbytes() == 4*3 * elem_size
all(s != 0 for s in y.stride())   # True
x.data_ptr() != y.data_ptr()       # True — different storage
```

**Why `.contiguous()` after a broadcast.** Many kernels (e.g. `view`, MKL/cuDNN-backed ops) require contiguous input. A zero-stride broadcast LOOKS like a tensor of the right shape but is actually a pinned 1-D buffer being indexed. `.contiguous()` is the canonical fix — it allocates fresh storage, copies the broadcasted values, and returns a stride-`(M, 1)` tensor.

**Storage size as a diagnostic.** Pre-contiguous, `x.storage().nbytes()` reflects only the SOURCE 1-D buffer (3 elements). Post-contiguous, it reflects the full materialized shape (12 elements). Comparing these is the cheap way to confirm 'yes, the broadcast was actually copied'.

### Exercise 2 — detect zero-stride view, materialize via .contiguous(), report stride + storage delta

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `.stride()` inspection to detect a zero-stride broadcast view, then call `.contiguous()` and report the stride change + storage size delta that confirm materialization.
> Keywords: stride, contiguous, broadcast, storage
> ```

**KCs targeted:** `detect-zero-stride-from-stride-tuple`, `contiguous-allocates-fresh-storage`

Implement `ex2_materialize_broadcast(x)`. The deepening variant of ex1.

Inputs:
- `x`: an arbitrary `torch.Tensor`. May or may not have a zero-stride axis.

Return a dict with EXACTLY these keys:

- `'has_zero_stride'`: `bool`, `True` iff any axis of `x.stride()` is `0`.
- `'stride_before'`: `tuple[int, ...]`, `tuple(x.stride())`.
- `'storage_nbytes_before'`: `int`, `x.untyped_storage().nbytes()` — the size of the underlying storage buffer in bytes (note: this is the source buffer, may be smaller than `x.numel() * elem_size` when broadcast).
- `'y'`: `torch.Tensor`, `x.contiguous()`. If `x` was already contiguous, `y` is `x` itself (PyTorch's contract — `x.contiguous()` returns `x` when already contiguous). Otherwise `y` is a fresh copy.
- `'stride_after'`: `tuple[int, ...]`, `tuple(y.stride())`.
- `'storage_nbytes_after'`: `int`, `y.untyped_storage().nbytes()`.
- `'is_contiguous_before'`: `bool`, `x.is_contiguous()`.
- `'is_contiguous_after'`: `bool`, `y.is_contiguous()` — must be `True` for any `y`.
- `'storage_shared'`: `bool`, `x.data_ptr() == y.data_ptr()` — `True` iff `y` is the same storage as `x` (i.e. no copy was needed).

Constraints:
- Do not mutate `x`.
- `stride_after` must contain NO zero strides for a tensor with more than one element.

In [ ]:
def ex2_materialize_broadcast(x):
    stride_before = tuple(x.stride())
    has_zero_stride = any(s == 0 for s in stride_before)
    storage_nbytes_before = x.untyped_storage().nbytes()
    is_contig_before = x.is_contiguous()

    y = x.contiguous()

    stride_after = tuple(y.stride())
    storage_nbytes_after = y.untyped_storage().nbytes()
    is_contig_after = y.is_contiguous()
    storage_shared = (x.data_ptr() == y.data_ptr())

    return {
        'has_zero_stride': has_zero_stride,
        'stride_before': stride_before,
        'storage_nbytes_before': storage_nbytes_before,
        'y': y,
        'stride_after': stride_after,
        'storage_nbytes_after': storage_nbytes_after,
        'is_contiguous_before': is_contig_before,
        'is_contiguous_after': is_contig_after,
        'storage_shared': storage_shared,
    }


<details><summary>Solution</summary>

```python
def ex2_materialize_broadcast(x):
    stride_before = tuple(x.stride())
    has_zero_stride = any(s == 0 for s in stride_before)
    storage_nbytes_before = x.untyped_storage().nbytes()
    is_contig_before = x.is_contiguous()

    y = x.contiguous()

    stride_after = tuple(y.stride())
    storage_nbytes_after = y.untyped_storage().nbytes()
    is_contig_after = y.is_contiguous()
    storage_shared = (x.data_ptr() == y.data_ptr())

    return {
        'has_zero_stride': has_zero_stride,
        'stride_before': stride_before,
        'storage_nbytes_before': storage_nbytes_before,
        'y': y,
        'stride_after': stride_after,
        'storage_nbytes_after': storage_nbytes_after,
        'is_contiguous_before': is_contig_before,
        'is_contiguous_after': is_contig_after,
        'storage_shared': storage_shared,
    }
```

**`x.contiguous()` returns `x` when already contiguous.** This is documented PyTorch behavior — no-op cost, no allocation. The `storage_shared` check via `data_ptr()` lets you detect whether a copy actually happened, which is what matters for memory budgeting.

**`untyped_storage().nbytes()` over `numel() * elem_size`.** For a broadcast view, `numel()` reports the broadcasted shape (e.g. 12 for a `(4, 3)` expanded view) but storage is only the source buffer (3 elements). `untyped_storage().nbytes()` reports the TRUE allocation. Comparing before/after gives the real memory delta.

**Transpose is non-contiguous but stride-positive.** It permutes the strides, doesn't zero them. So `has_zero_stride=False` while `is_contiguous=False`. The two diagnostics are independent and `.contiguous()` fixes BOTH (it always returns a stride-(M·N, N, 1)-style tensor).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()